# Setup NemoClaw

This notebook is meant to run **on the host itself** — a [Brev](https://brev.nvidia.com) instance or any other Linux host. It walks through the host-side flow for **NemoClaw**: configure or reuse a sandbox, pin the tested Docker package versions, install NemoClaw and onboard the sandbox with your chosen agent harness (OpenClaw or Hermes), apply the VSS policy, install the VSS skills, upload the workspace bootstrap docs, register or confirm the VSS Orchestrator MCP path, configure optional OpenClaw webhooks, then optionally verify the live sandbox, active policy, webhooks, and installed workspace docs.

A few steps are Brev-specific — the notebook reads secure-link FQDNs from `BREV_ENVIRONMENT_CONTEXT_PATH` (default `/etc/brev/environment-context.json`) and the generated remote UI link — and are called out where they apply. On other platforms, install the prerequisites yourself and reach the agent UI over your own networking (e.g. the SSH tunnel shown in section 3.7).

Once NemoClaw is up (and you have opened the Agent UI in section 3.7), continue with the companion notebook **`deploy_vss_orchestrator.ipynb`** to prepare the host, start the VSS Orchestrator MCP server, and deploy/manage VSS from the agent UI.

**Required prerequisites**

- Run this notebook with **Python 3.11 or newer**.
- Make sure the intended VSS checkout is the one resolved by `VSS_REPO_DIR`. By default this notebook uses `~/video-search-and-summarization`; set the `VSS_REPO_DIR` environment variable before launching Jupyter if your checkout lives elsewhere or the host has multiple clones.
- If you are re-onboarding a host that previously ran an older OpenShell gateway, follow the NemoClaw sandbox lifecycle upgrade guidance: <https://docs.nvidia.com/nemoclaw/manage-sandboxes/lifecycle>.

**What this notebook covers**

- Choose an agent model provider (which sets the required API key) and set notebook options.
- Run preflight checks for the local repo, scripts, policy file, and host prerequisites, then pin Docker to the tested package versions.
- Create or reuse the NemoClaw sandbox, configure the chosen agent model provider, apply the VSS policy, install VSS skills, upload workspace docs, register or confirm VSS Orchestrator MCP access, and bake the agent UI origin via `CHAT_UI_URL` at onboard.
- Open the Agent UI (OpenClaw shown as an example); the companion notebook then verifies the agent and deploys VSS.
- *(Optional)* Verify the live sandbox state, active policy metadata, OpenClaw webhooks (if enabled), and installed workspace docs.

**Security:** prefer `NVIDIA_API_KEY` from environment variables or your platform's secret store (e.g. Brev secrets). Do **not** commit notebook outputs that contain credentials or live access tokens.


## 1. Settings

Configure the notebook in two parts:

1. **Initialize provider variables** — seed the agent model-provider variables before you choose a provider.
2. **Choose ONE agent model provider** — pick exactly one of (a) a SOTA cloud model, (b) a locally-hosted OpenAI-compatible model, or (c) a model from build.nvidia.com.

> Run section 1.1, then run **exactly one** of the (a) / (b) / (c) cells. Section 1.3 holds advanced defaults you can usually leave alone.

<span style="color:red"><strong>Important:</strong> set at least one of <code>NVIDIA_API_KEY</code> / <code>COMPATIBLE_API_KEY</code> via the provider cell you pick. Credentials can also come from the environment or your platform's secret store (e.g. Brev secrets).</span>

### 1.1 Initialize agent model-provider variables

Run this cell first. It seeds the four agent model-provider variables to empty so that whichever **one** of the (a)/(b)/(c) cells you run in section 1.2 works on its own.


In [ ]:
# ================== Agent model vars (set by ONE of (a)/(b)/(c) in 1.2) ==================
NVIDIA_API_KEY = NEMOCLAW_ENDPOINT_URL = NEMOCLAW_MODEL = COMPATIBLE_API_KEY = ""

### 1.2 Choose ONE agent model provider

Run **exactly one** of the cells below. 

| Option | When to use | Sets |
|---|---|---|
| **(a) SOTA cloud model** | Best agent quality (Claude Opus, GPT-5, …) via any OpenAI-compatible cloud API. | `NEMOCLAW_ENDPOINT_URL`, `NEMOCLAW_MODEL`, `COMPATIBLE_API_KEY` |
| **(b) Local OpenAI-compatible model** | Self-hosted on this box or LAN | `NEMOCLAW_ENDPOINT_URL`, `NEMOCLAW_MODEL`, `COMPATIBLE_API_KEY` |
| **(c) build.nvidia.com NVIDIA-hosted model** | Default path — uses NVIDIA's hosted Nemotron via `integrate.api.nvidia.com`. | `NVIDIA_API_KEY`, optionally `NEMOCLAW_MODEL` |


#### (a) SOTA cloud model — *recommended for best agent quality*



In [ ]:
# (a) SOTA cloud model — fill in these three values, then run.
#

NEMOCLAW_ENDPOINT_URL = ""  # OpenAI-compatible base URL, e.g. "https://api.anthropic.com/v1/"
NEMOCLAW_MODEL        = ""  # Model id at that endpoint, e.g. "claude-opus-4-6"
COMPATIBLE_API_KEY    = ""  # Bearer token (sk-ant-..., sk-proj-..., etc.)

#### (b) Local OpenAI-compatible model — *self-hosted / air-gapped*

Point the agent at a local OpenAI-compatible server you've already started on this host or your LAN. For example, a downloadable NIM from build.nvidia.com


> `COMPATIBLE_API_KEY` is technically required by the installer (it errors if blank), but most local servers ignore the value — any non-empty placeholder works.
> From inside the NemoClaw sandbox, the host is reachable as `host.openshell.internal`. If your server binds only to `127.0.0.1` on the host, rebind it to `0.0.0.0` — the sandbox cannot reach a loopback-only socket via `host.openshell.internal`.

In [ ]:
# (b) Local OpenAI-compatible model — fill in to match your local server, then run.
NEMOCLAW_ENDPOINT_URL = "http://host.openshell.internal:8000/v1"
NEMOCLAW_MODEL        = "nvidia/nemotron-3-super-120b-a12b"  # the id your local server reports
COMPATIBLE_API_KEY    = "EMPTY"                              # most local servers ignore this; must be non-empty


#### (c) build.nvidia.com NVIDIA-hosted model — *default, zero-setup*

Uses NVIDIA's hosted model catalog via `integrate.api.nvidia.com`. Only `NVIDIA_API_KEY` is required; `NEMOCLAW_MODEL` defaults to `nvidia/nemotron-3-super-120b-a12b` inside the installer if left blank.

> Get a key at <https://build.nvidia.com> (format `nvapi-...`). To use a different hosted model, set `NEMOCLAW_MODEL` to its build.nvidia.com id (e.g. `nvidia/llama-3.3-nemotron-super-49b-v1.5`).


In [ ]:
# (c) build.nvidia.com — set NVIDIA_API_KEY; clear the custom-endpoint vars so the installer picks NEMOCLAW_PROVIDER=build.
NVIDIA_API_KEY = ""  # "nvapi-..." from https://build.nvidia.com
NEMOCLAW_MODEL = "qwen/qwen3.5-122b-a10b"  # leave blank to use (nvidia/nemotron-3-super-120b-a12b), or set a build.nvidia.com model id

### 1.3 Advanced settings (defaults — usually leave alone)

Pinned installer ref, sandbox agent harness, the inference API proxy toggle, and agent webhook plumbing. `AGENT_RUNTIME` accepts only `openclaw` or `hermes` and can be overridden by the `AGENT_RUNTIME` env var. `AGENT_DASHBOARD_PORT` is derived (default `18789`; override via `NEMOCLAW_DASHBOARD_PORT`). `BREV_ENVIRONMENT_CONTEXT_PATH` is derived (default `/etc/brev/environment-context.json`) — override via that env var only if the context file lives elsewhere.

`ORCHESTRATOR_ENABLE_HTTPS` only picks the scheme of the orchestrator MCP URL registered with the sandbox in section 3.5; the server itself is configured and started in `deploy_vss_orchestrator.ipynb`. Set the same value in both notebooks (or override via the `ORCHESTRATOR_ENABLE_HTTPS` env var).


In [ ]:
import os
import subprocess
from pathlib import Path


# ================== Default Nemoclaw settings ==================
# NemoClaw installer pin. Must be a release that ships the canonical
# `nemoclaw sandbox {policy add, skill install, config set, upload}`
# subcommands.
NEMOCLAW_INSTALL_REF = "v0.0.80"
# Sandbox agent harness: "openclaw" or "hermes". Override via the AGENT_RUNTIME env var.
AGENT_RUNTIME = "hermes"
# Enable SSRF detection for NEMOCLAW_ENDPOINT_URL and host proxy startup.
NEMOCLAW_INFERENCE_PROXY = True
AGENT_HOOKS_ENABLED = True     # agent webhooks (/hooks)
AGENT_HOOKS_PATH = "/hooks"
# Must match ORCHESTRATOR_ENABLE_HTTPS in deploy_vss_orchestrator.ipynb.
ORCHESTRATOR_ENABLE_HTTPS = False


# ================== Derived (no need to touch) ==================

HOME_DIR = Path.home().resolve()
NVIDIA_API_KEY = (NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY", "")).strip()
# Brev context JSON (secure-link FQDNs + environment id). Override via env if needed.
BREV_ENVIRONMENT_CONTEXT_PATH = os.environ.get("BREV_ENVIRONMENT_CONTEXT_PATH", "/etc/brev/environment-context.json").strip()
os.environ["BREV_ENVIRONMENT_CONTEXT_PATH"] = BREV_ENVIRONMENT_CONTEXT_PATH
# Agent UI port (OpenClaw / Hermes). Override via NEMOCLAW_DASHBOARD_PORT if needed.
AGENT_DASHBOARD_PORT = int(os.environ.get("NEMOCLAW_DASHBOARD_PORT", "18789") or "18789")
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", HOME_DIR / "video-search-and-summarization")).resolve()
NEMOCLAW_REPO_DIR = Path(os.environ.get("NEMOCLAW_REPO_DIR", HOME_DIR / "NemoClaw")).resolve()
INFERENCE_API_PROXY_PORT = int(os.environ.get("INFERENCE_API_PROXY_PORT", "18080"))
AGENT_RUNTIME = (os.environ.get("AGENT_RUNTIME") or AGENT_RUNTIME).strip().lower()
DEPLOY_SCRIPTS_DIR = VSS_REPO_DIR / "deploy" / "docker" / "scripts"
NEMOCLAW_PROVIDER = "custom" if NEMOCLAW_ENDPOINT_URL else "build"
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
SKILLS_DIR = VSS_REPO_DIR / "skills"
WORKSPACE_VARIANT = os.environ.get("AGENT_PLUGIN_VARIANT", "nemoclaw").strip() or "nemoclaw"
WORKSPACE_DIR = (VSS_REPO_DIR / ".openclaw" / "workspace").resolve()
WORKSPACE_REMOTE_DIR = "/sandbox" if AGENT_RUNTIME == "hermes" else "/sandbox/.openclaw/workspace"
SANDBOX_CONFIG_PATH = "/sandbox/.openclaw/openclaw.json"
AGENT_HOOKS_TOKEN = subprocess.check_output(["openssl", "rand", "-hex", "32"], text=True).strip() if AGENT_HOOKS_ENABLED else ""
INFERENCE_API_PROXY_PATH = DEPLOY_SCRIPTS_DIR / "nemoclaw" / "inference-api-proxy.py"
ORCHESTRATOR_MCP_HELPER_PATH = DEPLOY_SCRIPTS_DIR / "orchestrator_mcp_helper.py"
BREV_UTIL_PATH = VSS_REPO_DIR / "services" / "agent" / "packages" / "vss_agents" / "src" / "vss_agents" / "orchestrator" / "brev_util.py"
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9988"))
HOST_INTERNAL_ALIAS = os.environ.get("HOST_INTERNAL_ALIAS", "host.openshell.internal").strip()
ORCHESTRATOR_ENABLE_HTTPS = (os.environ.get("ORCHESTRATOR_ENABLE_HTTPS", str(ORCHESTRATOR_ENABLE_HTTPS)).strip().lower() == "true")
MCP_SCHEME = "https" if ORCHESTRATOR_ENABLE_HTTPS else "http"
ORCHESTRATOR_MCP_URL = f"{MCP_SCHEME}://{HOST_INTERNAL_ALIAS}:{MCP_PORT}/mcp"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo").strip()
NEMOCLAW_INSTALL_REF = os.environ.get("NEMOCLAW_INSTALL_REF", NEMOCLAW_INSTALL_REF).strip()

print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("NEMOCLAW_INSTALL_REF:", NEMOCLAW_INSTALL_REF)
print("\nHOME_DIR:", HOME_DIR)
print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("NEMOCLAW_REPO_DIR:", NEMOCLAW_REPO_DIR)
print("POLICY_PATH:", POLICY_PATH)
print("SKILLS_DIR:", SKILLS_DIR)
print("WORKSPACE_DIR:", WORKSPACE_DIR)
print("WORKSPACE_REMOTE_DIR:", WORKSPACE_REMOTE_DIR)
print("INFERENCE_API_PROXY_PATH:", INFERENCE_API_PROXY_PATH)
print("ORCHESTRATOR_MCP_HELPER_PATH:", ORCHESTRATOR_MCP_HELPER_PATH)
print("BREV_UTIL_PATH:", BREV_UTIL_PATH)
print("AGENT_RUNTIME:", AGENT_RUNTIME)
print("BREV_ENVIRONMENT_CONTEXT_PATH:", BREV_ENVIRONMENT_CONTEXT_PATH)
print("AGENT_DASHBOARD_PORT:", AGENT_DASHBOARD_PORT)
print("WORKSPACE_VARIANT:", WORKSPACE_VARIANT)
print("HOST_INTERNAL_ALIAS:", HOST_INTERNAL_ALIAS)
print("ORCHESTRATOR_MCP_URL:", ORCHESTRATOR_MCP_URL)
print("ORCHESTRATOR_ENABLE_HTTPS:", ORCHESTRATOR_ENABLE_HTTPS)
print("NEMOCLAW_PROVIDER:", NEMOCLAW_PROVIDER)
print("NEMOCLAW_INFERENCE_PROXY:", NEMOCLAW_INFERENCE_PROXY)
if NEMOCLAW_ENDPOINT_URL:
    print("NEMOCLAW_ENDPOINT_URL:", NEMOCLAW_ENDPOINT_URL)
    print("NEMOCLAW_MODEL:", NEMOCLAW_MODEL)
    print("COMPATIBLE_API_KEY set:", bool(COMPATIBLE_API_KEY))
print("Agent hooks enabled:", AGENT_HOOKS_ENABLED)
if AGENT_HOOKS_ENABLED:
    print("Agent hooks path:", AGENT_HOOKS_PATH)
    print("Agent hooks token set:", bool(AGENT_HOOKS_TOKEN))
print("NVIDIA_API_KEY set:", bool(NVIDIA_API_KEY))


## 2. Preflight

Run the next cell to confirm the expected keys, files, commands are present on the host.


In [ ]:
import importlib.util
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"

helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

brev_util_spec = importlib.util.spec_from_file_location("vss_brev_util", BREV_UTIL_PATH)
if brev_util_spec is None or brev_util_spec.loader is None:
    raise ImportError(f"Could not load brev_util from {BREV_UTIL_PATH}")
brev_util = importlib.util.module_from_spec(brev_util_spec)
brev_util_spec.loader.exec_module(brev_util)

# Needed later: 3.1 (UI origin) and 3.7 (gateway container lookup).
brev_environment_id = brev_util.brev_environment_id
brev_secure_link_fqdn = brev_util.brev_secure_link_fqdn
resolve_openshell_gateway_container = orchestrator_mcp_helper.resolve_openshell_gateway_container


def agent_provider_configured() -> bool:
    if NEMOCLAW_PROVIDER == "build":
        return bool(NVIDIA_API_KEY)
    if NEMOCLAW_PROVIDER == "custom":
        return bool(NEMOCLAW_ENDPOINT_URL and NEMOCLAW_MODEL and COMPATIBLE_API_KEY)
    return False


required_checks = {
    "Agent model provider configured": agent_provider_configured(),
    "NEMOCLAW_INSTALL_REF set": bool(NEMOCLAW_INSTALL_REF),
    "vss_nemoclaw_policy.yaml": POLICY_PATH.is_file(),
    "skills/": SKILLS_DIR.is_dir(),
    "workspace bootstrap dir": WORKSPACE_DIR.is_dir(),
    "orchestrator_mcp_helper.py": ORCHESTRATOR_MCP_HELPER_PATH.is_file(),
    "brev_util.py": BREV_UTIL_PATH.is_file(),
    # host commands
    "docker": shutil.which("docker") is not None,
    "python3": shutil.which("python3") is not None,
    "curl": shutil.which("curl") is not None,
}

if NEMOCLAW_PROVIDER == "custom":
    required_checks["NEMOCLAW_ENDPOINT_URL set"] = bool(NEMOCLAW_ENDPOINT_URL)
    required_checks["NEMOCLAW_MODEL set"] = bool(NEMOCLAW_MODEL)
    required_checks["COMPATIBLE_API_KEY set"] = bool(COMPATIBLE_API_KEY)
elif NEMOCLAW_PROVIDER == "build":
    required_checks["NVIDIA_API_KEY set (build provider)"] = bool(NVIDIA_API_KEY)

if AGENT_HOOKS_ENABLED:
    required_checks["AGENT_HOOKS_TOKEN set"] = bool(AGENT_HOOKS_TOKEN)
    required_checks["AGENT_HOOKS_PATH set"] = bool(AGENT_HOOKS_PATH)

optional_checks = {
    "NemoClaw repo checkout": NEMOCLAW_REPO_DIR.is_dir(),
}

for label, ok in required_checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")


### 2.1 Pin Docker version

Pin Docker CE + plugins + containerd.io to a known-good combination (CE **29.4.3**, buildx **0.33.0**, compose **5.1.3**, containerd **2.2.3**) **before** section 3 brings up the NemoClaw sandbox — a docker-ce downgrade restarts dockerd and would disrupt live sandbox containers if it ran later in the notebook. `apt-mark hold` prevents drift back to newer versions for the rest of the deployment.

The cell first reads the installed Docker Engine version: if it already falls in the tested range **[28.3.3, 29.5.0)** the version downgrade is **skipped** — re-pinning to an exact version the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64) would fail with *version not found* for no benefit. The packages are still `apt-mark hold`-ed at their current versions so the box can't drift past the tested range mid-run. Safe to re-run.

In [ ]:
%%bash
# Pin Docker CE + plugins + containerd.io to a known-good combination, but
# ONLY when the host's Docker is outside the tested range. Some Brev
# launchables ship a newer Docker than the VSS deploy profiles are tested
# against; pin explicitly so compose/buildx incompatibilities don't surface
# mid-deployment.
#
# When the installed Docker already falls in [28.3.3, 29.5.0) the version
# downgrade is skipped: re-pinning to an exact epoch-versioned package that
# the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64)
# fails with "version not found" for no benefit. The in-range packages are
# still held so the box can't drift past the tested range mid-notebook.
# Idempotent -- safe to re-run.

set -euo pipefail

# Tested Docker Engine range -- keep in sync with the VSS launchable prereq check.
MIN_DOCKER_VERSION="28.3.3"
MAX_DOCKER_VERSION="29.5.0"

# Packages frozen with `apt-mark hold` so unattended-upgrades / later
# `apt-get install` calls can't drift the box before the notebook finishes.
HOLD_PKGS="docker-ce docker-ce-cli docker-buildx-plugin docker-compose-plugin containerd.io"

version_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]; }
version_lt() { [ "$1" != "$2" ] && [ "$(printf '%s\n%s\n' "$1" "$2" | sort -V | head -n1)" = "$1" ]; }

DOCKER_VERSION="$(docker version --format '{{.Server.Version}}' 2>/dev/null || true)"
if [ -n "$DOCKER_VERSION" ] \
   && version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
   && version_lt "$DOCKER_VERSION" "$MAX_DOCKER_VERSION"; then
  echo "Docker $DOCKER_VERSION is within the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); skipping the Docker version pin."
  # No downgrade needed, but still hold the in-range packages at their
  # current versions so unattended-upgrades / later apt-get calls can't drift
  # the box past the tested range for the remainder of the notebook.
  sudo apt-mark hold $HOLD_PKGS
  exit 0
fi

if [ -n "$DOCKER_VERSION" ]; then
  echo "Docker $DOCKER_VERSION is outside the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); pinning to known-good versions."
else
  echo "Could not read the installed Docker version; pinning to known-good versions."
fi

# Read distro info from /etc/os-release (always present on Ubuntu; minimal
# images don't ship `lsb_release`).
. /etc/os-release
DISTRO="${VERSION_ID}"
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME}}"

# Versions hard-coded to what shipped alongside docker-ce 29.4.3 on the
# Docker apt repo (verified against download.docker.com + upstream GitHub
# release timestamps). When bumping DOCKER_CE_VER, bump these four together.
DOCKER_CE_VER="5:29.4.3-1~ubuntu.${DISTRO}~${CODENAME}"
BUILDX_VER="0.33.0-1~ubuntu.${DISTRO}~${CODENAME}"
COMPOSE_VER="5.1.3-1~ubuntu.${DISTRO}~${CODENAME}"
CONTAINERD_VER="2.2.3-1~ubuntu.${DISTRO}~${CODENAME}"

# Refresh the APT cache first -- without this, the specific epoch-versioned
# package may not be in the local index and the install would fail with
# version-not-found before any pinning takes effect.
sudo apt-get update -qq

sudo DEBIAN_FRONTEND=noninteractive apt-get install -y \
  --allow-downgrades \
  -o Dpkg::Options::=--force-confdef \
  -o Dpkg::Options::=--force-confold \
  docker-ce="$DOCKER_CE_VER" \
  docker-ce-cli="$DOCKER_CE_VER" \
  docker-buildx-plugin="$BUILDX_VER" \
  docker-compose-plugin="$COMPOSE_VER" \
  containerd.io="$CONTAINERD_VER"

# Hold so unattended-upgrades / later `apt-get install` calls don't drift
# the box back to newer versions before the rest of the notebook runs.
sudo apt-mark hold $HOLD_PKGS

## 3. Install and Configure NemoClaw for VSS skills

Sections 3.1–3.6 install and configure the sandbox using canonical NemoClaw / OpenShell commands. Run the cells in order — each step is idempotent and safe to re-run. Section 3.7 opens the Agent UI, and section 3.8 is an optional post-setup verification.

| Step | What it does |
|---|---|
| 3.1 | Install NemoClaw (pinned `NEMOCLAW_INSTALL_REF`) and create the sandbox |
| 3.2 | Apply the VSS sandbox policy |
| 3.3 | Install the VSS skills |
| 3.4 | Upload the workspace bootstrap docs |
| 3.5 | Register or confirm the VSS Orchestrator MCP path |
| 3.6 | Configure optional webhooks |
| 3.7 | Open the Agent UI |
| 3.8 | *(Optional)* Verify sandbox, policy, workspace, and webhooks |


### 3.1 Install NemoClaw and create the sandbox

Exports the NemoClaw configuration (including `CHAT_UI_URL`, the dashboard origin for UI access) and installs NemoClaw at the pinned `NEMOCLAW_INSTALL_REF`. The non-interactive installer also creates and onboards the sandbox. Takes a few minutes; output streams live.

When `NEMOCLAW_ENDPOINT_URL`'s host resolves to a non-public address, this step automatically starts the bundled host proxy and points NemoClaw at it, avoiding NemoClaw's SSRF rejection. Set the advanced setting `NEMOCLAW_INFERENCE_PROXY = False` to disable this behavior; it defaults to `True`.


In [ ]:
import ipaddress
import json
import os
import re
import shlex
import shutil
import socket
import subprocess
import sys
import time
from pathlib import Path
from urllib.parse import urlsplit


if not isinstance(NEMOCLAW_INFERENCE_PROXY, bool):
    raise TypeError("NEMOCLAW_INFERENCE_PROXY must be True or False")


def _resolved_addresses(host):
    try:
        return sorted(
            {
                info[4][0]
                for info in socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            }
        )
    except socket.gaierror as exc:
        print(f"Could not resolve {host}; leaving the endpoint unchanged: {exc}", flush=True)
        return []


def _proxy_port_is_open():
    try:
        with socket.create_connection(("127.0.0.1", INFERENCE_API_PROXY_PORT), timeout=0.5):
            return True
    except OSError:
        return False


def _ensure_inference_api_proxy(upstream_host):
    if _proxy_port_is_open():
        print(f"Inference API proxy already listening on port {INFERENCE_API_PROXY_PORT}.", flush=True)
        return
    if not INFERENCE_API_PROXY_PATH.is_file():
        raise FileNotFoundError(
            f"NemoClaw needs the inference API proxy, but it was not found at {INFERENCE_API_PROXY_PATH}. "
            "Set INFERENCE_API_PROXY_PATH to inference-api-proxy.py."
        )

    proxy_env = os.environ.copy()
    proxy_env["INFERENCE_API_UPSTREAM"] = upstream_host
    proxy_env["INFERENCE_API_PROXY_HOST"] = "0.0.0.0"
    proxy_env["INFERENCE_API_PROXY_PORT"] = str(INFERENCE_API_PROXY_PORT)
    log_path = Path("/tmp/inference-api-proxy.log")
    with log_path.open("ab") as log_file:
        process = subprocess.Popen(
            [sys.executable, str(INFERENCE_API_PROXY_PATH)],
            env=proxy_env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    for _ in range(50):
        if _proxy_port_is_open():
            print(
                f"Started {INFERENCE_API_PROXY_PATH} on port {INFERENCE_API_PROXY_PORT} "
                f"-> {upstream_host} (log: {log_path}).",
                flush=True,
            )
            return
        if process.poll() is not None:
            break
        time.sleep(0.1)
    raise RuntimeError(f"Inference API proxy failed to start; inspect {log_path}")


_effective_nemoclaw_endpoint_url = NEMOCLAW_ENDPOINT_URL
_inference_proxy_active = False
_endpoint = urlsplit(NEMOCLAW_ENDPOINT_URL) if NEMOCLAW_ENDPOINT_URL else None
_endpoint_host = _endpoint.hostname if _endpoint else None
if NEMOCLAW_INFERENCE_PROXY and _endpoint_host:
    _endpoint_addresses = _resolved_addresses(_endpoint_host)
    _has_non_public_address = any(
        not ipaddress.ip_address(address).is_global for address in _endpoint_addresses
    )
    if _has_non_public_address:
        print(
            f"{_endpoint_host} resolves to {_endpoint_addresses}; "
            "using the host proxy to avoid NemoClaw SSRF rejection.",
            flush=True,
        )
        _ensure_inference_api_proxy(_endpoint_host)
        _endpoint_path = _endpoint.path.rstrip("/") or "/v1"
        _effective_nemoclaw_endpoint_url = (
            f"http://host.openshell.internal:{INFERENCE_API_PROXY_PORT}{_endpoint_path}"
        )
        _inference_proxy_active = True

# Fail fast if no model provider is configured (section 1.2).
if NEMOCLAW_PROVIDER == "build" and not NVIDIA_API_KEY:
    raise RuntimeError(
        "No agent provider configured. Pick one and fill in its values:\n"
        "  - option (a) SOTA cloud / (b) local: set NEMOCLAW_ENDPOINT_URL, NEMOCLAW_MODEL, and COMPATIBLE_API_KEY in the (a) or (b) cell\n"
        "  - option (c) build.nvidia.com: set NVIDIA_API_KEY in the (c) cell"
    )
if NEMOCLAW_PROVIDER == "custom" and not COMPATIBLE_API_KEY:
    raise RuntimeError(
        "COMPATIBLE_API_KEY is required for the custom OpenAI-compatible provider (options a/b). "
        "Set it in the (a) or (b) cell (any non-empty placeholder works for most local servers)."
    )
if NEMOCLAW_PROVIDER == "custom" and not NEMOCLAW_MODEL:
    raise RuntimeError(
        "NEMOCLAW_MODEL is required when NEMOCLAW_ENDPOINT_URL is set. "
        "If you ran more than one of (a)/(b)/(c), re-run just the (a) or (b) cell you want to use."
    )

# NemoClaw reads its configuration from environment variables.
env = os.environ.copy()
env["VSS_REPO_DIR"] = str(VSS_REPO_DIR)
env["NEMOCLAW_SANDBOX_NAME"] = NEMOCLAW_SANDBOX_NAME
env["NEMOCLAW_INSTALL_REF"] = NEMOCLAW_INSTALL_REF
env["NEMOCLAW_PROVIDER"] = NEMOCLAW_PROVIDER
env["NEMOCLAW_NON_INTERACTIVE"] = "1"
env["NEMOCLAW_ACCEPT_THIRD_PARTY_SOFTWARE"] = "1"
env["NEMOCLAW_AGENT"] = AGENT_RUNTIME
env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if NEMOCLAW_MODEL:
    env["NEMOCLAW_MODEL"] = NEMOCLAW_MODEL
if NEMOCLAW_PROVIDER == "custom":
    env["NEMOCLAW_ENDPOINT_URL"] = _effective_nemoclaw_endpoint_url
    env["COMPATIBLE_API_KEY"] = COMPATIBLE_API_KEY
    print(
        "Custom OpenAI-compatible provider: "
        f"{_effective_nemoclaw_endpoint_url} model={NEMOCLAW_MODEL} "
        f"key set={bool(COMPATIBLE_API_KEY)}"
    )
# CHAT_UI_URL must be correct at onboard time: the sandbox derives the UI
# allowed origins and the 0.0.0.0 forward bind from it (gateway.* is read-only later).
_chat_fqdn = brev_secure_link_fqdn(AGENT_DASHBOARD_PORT)
if _chat_fqdn:
    env["CHAT_UI_URL"] = f"https://{_chat_fqdn}"
    print("CHAT_UI_URL:", env["CHAT_UI_URL"])
else:
    print(
        f"WARNING: no FQDN for port {AGENT_DASHBOARD_PORT} in {BREV_ENVIRONMENT_CONTEXT_PATH}; "
        "onboard will proceed without baking a remote UI origin."
    )
os.environ.update(env)  # export for the ! commands below


def _version_of(text):
    match = re.search(r"v?(\d+\.\d+\.\d+)", text or "")
    return match.group(1) if match else None


install_ref = NEMOCLAW_INSTALL_REF.strip()
if not install_ref:
    raise RuntimeError("NEMOCLAW_INSTALL_REF must not be empty.")
install_url = f"https://raw.githubusercontent.com/NVIDIA/NemoClaw/{install_ref}/install.sh"


In [ ]:
local_bin = HOME_DIR / ".local" / "bin"
if str(local_bin) not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = f"{local_bin}{os.pathsep}{os.environ.get('PATH', '')}"

agent_label = "NemoHermes" if AGENT_RUNTIME == "hermes" else "NemoClaw"
agent_cli = "nemohermes" if AGENT_RUNTIME == "hermes" else "nemoclaw"
onboard_cmd = "nemohermes onboard --non-interactive" if AGENT_RUNTIME == "hermes" else f"nemoclaw onboard --non-interactive --agent {AGENT_RUNTIME}"
installed = None
if shutil.which(agent_cli):
    _ver = !{agent_cli} --version 2>&1
    installed = _version_of(" ".join(_ver))

if installed and installed == _version_of(install_ref):
    print(f"{agent_label} {installed} already installed, skipping installer.", flush=True)
else:
    print(f"Installing {agent_label} {install_ref} (takes a few minutes)...", flush=True)
    !cd ~ && bash -o pipefail -c "curl -fsSL {install_url} | bash"
    install_exit_code = _exit_code
    if install_exit_code != 0:
        if not shutil.which(agent_cli):
            raise AssertionError(f"{agent_label} install failed")
        print(
            f"{agent_label} installer exited non-zero after installing {agent_cli}; "
            "continuing with explicit onboarding.",
            flush=True,
        )

!{agent_cli} --version
assert _exit_code == 0, f"{agent_cli} is not runnable after install"

!openshell sandbox get {NEMOCLAW_SANDBOX_NAME} >/dev/null 2>&1
if _exit_code != 0:
    print(
        f"No sandbox {NEMOCLAW_SANDBOX_NAME!r} yet — onboarding "
        f"(agent={AGENT_RUNTIME}, takes several minutes)...",
        flush=True,
    )
    !cd ~ && {onboard_cmd}
    if _exit_code != 0 and AGENT_RUNTIME == "hermes":
        print("Hermes onboard failed; retrying with refreshed sandbox base-image resolution...", flush=True)
        os.environ.pop("NEMOCLAW_HERMES_SANDBOX_BASE_IMAGE_REF", None)
        os.environ["NEMOCLAW_SANDBOX_BASE_IMAGE_REFRESH"] = "1"
        !cd ~ && NEMOCLAW_SANDBOX_BASE_IMAGE_REFRESH=1 {onboard_cmd}
    if _exit_code != 0 and AGENT_RUNTIME == "hermes":
        print("Hermes onboard still failed; retrying with --fresh and refreshed sandbox base-image resolution...", flush=True)
        !cd ~ && NEMOCLAW_SANDBOX_BASE_IMAGE_REFRESH=1 nemohermes onboard --fresh --non-interactive
    assert _exit_code == 0, f"{agent_cli} onboard failed"
print(f"Sandbox {NEMOCLAW_SANDBOX_NAME!r} ready.", flush=True)


### 3.2 Apply the VSS sandbox policy

Merges `assets/vss_nemoclaw_policy.yaml` into the base OpenShell policy.


In [ ]:
if not POLICY_PATH.is_file():
    raise FileNotFoundError(f"Missing policy file: {POLICY_PATH}")
if AGENT_RUNTIME == "hermes":
    !nemohermes {NEMOCLAW_SANDBOX_NAME} policy-add --from-file {POLICY_PATH} --yes
else:
    !nemoclaw sandbox policy add {NEMOCLAW_SANDBOX_NAME} --from-file {POLICY_PATH} --yes
assert _exit_code == 0, "policy add failed"


### 3.3 Install the VSS skills

Installs each `skills/<name>/` directory containing a `SKILL.md`.


In [ ]:
if not SKILLS_DIR.is_dir():
    raise FileNotFoundError(f"Missing skills dir: {SKILLS_DIR}")
skill_dirs = sorted({sp.parent for sp in SKILLS_DIR.glob("*/SKILL.md")})
if not skill_dirs:
    raise RuntimeError(f"No SKILL.md directories found under {SKILLS_DIR}")
for skill_dir in skill_dirs:
    if AGENT_RUNTIME == "hermes":
        !nemohermes {NEMOCLAW_SANDBOX_NAME} skill install {skill_dir}
    else:
        !nemoclaw sandbox skill install {NEMOCLAW_SANDBOX_NAME} {skill_dir}
    assert _exit_code == 0, f"skill install failed: {skill_dir.name}"
print(f"Installed {len(skill_dirs)} VSS skills.", flush=True)


### 3.4 Upload the workspace bootstrap docs

Uploads the shared workspace `.md` docs, then the `_<variant>` overlay (overlay last, so it wins).


In [ ]:
if not WORKSPACE_DIR.is_dir():
    print(f"No workspace dir at {WORKSPACE_DIR}; skipping workspace upload.", flush=True)
else:
    docs = sorted(WORKSPACE_DIR.glob("*.md"))
    overlay_dir = WORKSPACE_DIR / f"_{WORKSPACE_VARIANT}"
    if overlay_dir.is_dir():
        docs += sorted(overlay_dir.glob("*.md"))
    # Self-heal: an earlier upload form used a file-path dest, which the
    # directory-semantics transport turned into <name>.md/<name>.md nesting on
    # fresh sandboxes. Remove any directory-shaped *.md leftovers first.
    !openshell sandbox exec -n {NEMOCLAW_SANDBOX_NAME} -- sh -c "mkdir -p {WORKSPACE_REMOTE_DIR} && find {WORKSPACE_REMOTE_DIR} -mindepth 1 -maxdepth 1 -type d -name '*.md' -exec rm -rf '{{}}' ';'"
    for doc in docs:
        # dest is a DIRECTORY: the OpenShell transport does mkdir + tar-extract
        # into it; a file path here collides with an existing file of that name.
        if AGENT_RUNTIME == "hermes":
            !nemohermes {NEMOCLAW_SANDBOX_NAME} upload {doc} {WORKSPACE_REMOTE_DIR}/ >/dev/null
        else:
            !nemoclaw sandbox upload {NEMOCLAW_SANDBOX_NAME} {doc} {WORKSPACE_REMOTE_DIR}/ >/dev/null
        assert _exit_code == 0, f"upload failed: {doc.name}"
    print(f"Uploaded {len(docs)} workspace docs to {WORKSPACE_REMOTE_DIR}.", flush=True)


### 3.5 Register the VSS Orchestrator MCP

Registers the host-side VSS Orchestrator MCP at `ORCHESTRATOR_MCP_URL` — the `HOST_INTERNAL_ALIAS` address the sandbox can actually reach, on the scheme `ORCHESTRATOR_ENABLE_HTTPS` selects. Skipped if already registered, so if you flip the scheme after a first run, remove the old entry (`nemoclaw sandbox mcp <sandbox> remove vss_orchestrator`) before re-running.


In [ ]:
!nemoclaw sandbox mcp {NEMOCLAW_SANDBOX_NAME} status vss_orchestrator >/dev/null 2>&1
if _exit_code == 0:
    print("MCP server 'vss_orchestrator' already registered.", flush=True)
else:
    !nemoclaw sandbox mcp {NEMOCLAW_SANDBOX_NAME} add vss_orchestrator --url {ORCHESTRATOR_MCP_URL}
    assert _exit_code == 0, "mcp add failed"


### 3.6 Configure optional OpenClaw webhooks

Writes the OpenClaw webhook keys (`hooks.*`), then restarts the gateway so they take effect. If the managed restart hits `SUPERVISOR_UNAVAILABLE`, falls back to `nemoclaw sandbox recover`. Skipped for Hermes or when `AGENT_HOOKS_ENABLED` is off.


In [ ]:
# Only the webhook keys need config set:
# - gateway.* is off-limits (auth tokens) — UI allowedOrigins come from
#   CHAT_UI_URL baked at onboard (section 3.1);
# - workspace docs are uploaded in section 3.4.
# Write keys first, then restart separately. `config set --restart` can update
# the config and still exit non-zero with SUPERVISOR_UNAVAILABLE when the
# in-sandbox supervisor is gone; recover relaunches it.
config_sets = []
if AGENT_HOOKS_ENABLED and AGENT_RUNTIME != "hermes":
    config_sets.append(("hooks.enabled", "true"))
    config_sets.append(("hooks.path", AGENT_HOOKS_PATH or "/hooks"))
    if AGENT_HOOKS_TOKEN:
        config_sets.append(("hooks.token", AGENT_HOOKS_TOKEN))

if not config_sets:
    print("No sandbox config changes needed (webhooks disabled or Hermes runtime).", flush=True)
else:
    for _key, _value in config_sets:
        _quoted = shlex.quote(_value)  # JSON values contain spaces/quotes
        !nemoclaw sandbox config set {NEMOCLAW_SANDBOX_NAME} --key {_key} --value {_quoted} --config-accept-new-path
        assert _exit_code == 0, f"config set failed: {_key}"
    print("Restarting gateway to apply webhook config...", flush=True)
    !nemoclaw sandbox gateway restart {NEMOCLAW_SANDBOX_NAME}
    if _exit_code != 0:
        print("Managed gateway restart failed — falling back to sandbox recover...", flush=True)
        !nemoclaw sandbox recover {NEMOCLAW_SANDBOX_NAME}
        assert _exit_code == 0, "sandbox recover failed after webhook config"
    print("Sandbox config applied; gateway restarted.", flush=True)


### 3.7 Open the Agent UI

Open the **Agent UI** for the sandbox so you can chat with the agent and drive VSS from it.

> The link cell and screenshots use **OpenClaw as an example**. If you onboarded a different agent harness (`AGENT_RUNTIME`, e.g. Hermes), open that harness's UI instead and follow the equivalent steps.

Run the next cell to print a fresh **Agent UI** link, then open it in your browser.

<span style="color:red"><strong>Not on Brev?</strong> If you are accessing the UI from a different machine, open an SSH tunnel before opening the Agent web UI from below:<br/><code>ssh -L &lt;AGENT_DASHBOARD_PORT&gt;:127.0.0.1:&lt;AGENT_DASHBOARD_PORT&gt; &lt;user&gt;@&lt;nemoclaw-host&gt;</code><br/></span>


In [ ]:
import os
import subprocess


def fetch_gateway_token():
    result = subprocess.run(
        ["nemoclaw", "sandbox", "gateway", "token", NEMOCLAW_SANDBOX_NAME, "--quiet"],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()


gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container; no UI link generated.")

_chat_fqdn = brev_secure_link_fqdn(AGENT_DASHBOARD_PORT)

if _chat_fqdn:
    origin = f"https://{_chat_fqdn}"
else:
    origin = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}"
    nemoclaw_host = subprocess.run(
        ["hostname", "-I"], capture_output=True, text=True, check=True
    ).stdout.split()[0]
    ssh_user = os.environ.get("USER", "ubuntu")
    RED, RESET = "\033[31m", "\033[0m"
    print(f"{RED}Make sure an SSH tunnel is running on your laptop before opening the Agent web UI:{RESET}")
    print(f"{RED}  $ ssh -L {AGENT_DASHBOARD_PORT}:localhost:{AGENT_DASHBOARD_PORT} {ssh_user}@{nemoclaw_host}{RESET}")
    if BREV_ENVIRONMENT_CONTEXT_PATH:
        print(
            f"{RED}No FQDN for port {AGENT_DASHBOARD_PORT} in {BREV_ENVIRONMENT_CONTEXT_PATH}; "
            f"using localhost tunnel URL instead.{RESET}"
        )

if AGENT_RUNTIME == "hermes":
    print("Agent UI:", origin)
    print(f"Hermes terminal: nemohermes {NEMOCLAW_SANDBOX_NAME} connect")
else:
    token = fetch_gateway_token()
    agent_ui_url = f"{origin}/#token={token}" if token else origin
    print("Agent UI:", agent_ui_url)


### 3.8 [OPTIONAL] Verify sandbox, policy, workspace, and optional webhooks

The next cell checks:

- whether the sandbox exists,
- the current active sandbox policy metadata,
- the expected local policy path and version,
- whether OpenClaw webhooks accept a local test request when enabled,
- the installed OpenClaw skills/workspace files or Hermes top-level workspace docs.


In [ ]:
from datetime import datetime, timezone
import json
import re
import shlex
import shutil
import subprocess
import time
from pathlib import Path


def run(cmd, check=False, echo=True):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if echo:
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    return r


def _openshell_forward_running(port, sandbox_name):
    port = str(port)
    listing = subprocess.run(["openshell", "forward", "list"], capture_output=True, text=True)
    clean_listing = re.sub(r"\x1b\[[0-9;]*m", "", f"{listing.stdout}\n{listing.stderr}")
    for line in clean_listing.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0] == sandbox_name and parts[2] == port and parts[-1].lower() == "running":
            return True
    return False


def _openshell_forward_process_running(port, sandbox_name):
    port = str(port)
    markers = (
        f"openshell forward start {port} {sandbox_name}",
        f"openshell forward start --background {port} {sandbox_name}",
    )
    result = subprocess.run(["ps", "-eo", "args="], capture_output=True, text=True)
    if result.returncode != 0:
        return False
    return any(any(marker in line for marker in markers) for line in result.stdout.splitlines())


def _openshell_forward_owned_by_sandbox(port, sandbox_name):
    return _openshell_forward_running(port, sandbox_name) or _openshell_forward_process_running(port, sandbox_name)


def _dashboard_healthy(port):
    health_url = f"http://127.0.0.1:{port}/health"
    health = subprocess.run(["curl", "-fsS", health_url], capture_output=True, text=True)
    return health.returncode == 0


def ensure_dashboard_forward(port=None):
    if port is None:
        port = AGENT_DASHBOARD_PORT
    port = str(port)
    health_url = f"http://127.0.0.1:{port}/health"
    if _dashboard_healthy(port) and _openshell_forward_owned_by_sandbox(port, NEMOCLAW_SANDBOX_NAME):
        return
    if _dashboard_healthy(port):
        raise RuntimeError(
            f"Dashboard health is responding on {health_url}, but it is not a running OpenShell forward "
            f"for sandbox {NEMOCLAW_SANDBOX_NAME}. Stop that listener before continuing."
        )

    subprocess.run(["openshell", "forward", "stop", port, NEMOCLAW_SANDBOX_NAME], check=False, capture_output=True, text=True)
    if shutil.which("setsid"):
        subprocess.run(
            ["setsid", "-f", "openshell", "forward", "start", port, NEMOCLAW_SANDBOX_NAME],
            stdin=subprocess.DEVNULL,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        )
    else:
        subprocess.run(
            ["openshell", "forward", "start", "--background", port, NEMOCLAW_SANDBOX_NAME],
            capture_output=True,
            text=True,
            check=False,
        )

    for _ in range(10):
        if _dashboard_healthy(port) and _openshell_forward_owned_by_sandbox(port, NEMOCLAW_SANDBOX_NAME):
            return
        time.sleep(1)
    raise RuntimeError(f"Dashboard port-forward on {port} is not reachable at {health_url}")


print("Verification time (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S %Z"))
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Expected policy file:", POLICY_PATH)

policy_text = Path(POLICY_PATH).read_text()
preset_name_match = re.search(r"^\s+name:\s*(\S+)", policy_text, re.MULTILINE)
print("Expected preset name:", preset_name_match.group(1) if preset_name_match else "unknown")

sandbox_result = run(["openshell", "sandbox", "get", NEMOCLAW_SANDBOX_NAME], echo=False)
sandbox_summary = "\n".join(part for part in (sandbox_result.stdout, sandbox_result.stderr) if part)
sandbox_summary = re.sub(r"\x1b\[[0-9;]*m", "", sandbox_summary)
phase_match = re.search(r"Phase:\s+(.+)", sandbox_summary)
namespace_match = re.search(r"Namespace:\s+(.+)", sandbox_summary)
sandbox_id_match = re.search(r"Id:\s+(.+)", sandbox_summary)
print("Sandbox namespace:", namespace_match.group(1).strip() if namespace_match else "unknown")
print("Sandbox phase:", phase_match.group(1).strip() if phase_match else "unknown")
print("Sandbox id:", sandbox_id_match.group(1).strip() if sandbox_id_match else "unknown")

policy_result = run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME])

policy_summary = "\n".join(part for part in (policy_result.stdout, policy_result.stderr) if part)
status_match = re.search(r"Status:\s+(.+)", policy_summary)
active_match = re.search(r"Active:\s+(.+)", policy_summary)
hash_match = re.search(r"Hash:\s+([0-9a-f]+)", policy_summary)
print("Active policy status:", status_match.group(1).strip() if status_match else "unknown")
print("Active policy version:", active_match.group(1).strip() if active_match else "unknown")
print("Active policy hash:", hash_match.group(1) if hash_match else "unknown")

gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not WORKSPACE_DIR.is_dir():
    raise FileNotFoundError(f"Missing plugin workspace source dir: {WORKSPACE_DIR}")
expected_workspace_md = tuple(sorted(p.name for p in WORKSPACE_DIR.glob("*.md")))
if not expected_workspace_md:
    raise RuntimeError(f"No .md files found in {WORKSPACE_DIR}; cannot verify workspace install.")
print("Expected workspace .md files:", list(expected_workspace_md))

if AGENT_HOOKS_ENABLED and AGENT_RUNTIME != "hermes":
    if not AGENT_HOOKS_TOKEN:
        raise RuntimeError("AGENT_HOOKS_TOKEN is required to verify agent hooks.")

    ensure_dashboard_forward()
    hooks_path = "/" + AGENT_HOOKS_PATH.strip("/")
    hooks_url = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}{hooks_path}/agent"
    hooks_payload = json.dumps(
        {
            "name": "NemoClaw notebook verification",
            "message": "test",
        }
    )
    hooks_cmd = [
        "curl",
        "-sS",
        "-w", "\n%{http_code}",  # append the HTTP status as the last stdout line
        "-X",
        "POST",
        hooks_url,
        "-H",
        f"Authorization: Bearer {AGENT_HOOKS_TOKEN}",
        "-H",
        "Content-Type: application/json",
        "-d",
        hooks_payload,
    ]
    hooks_result = subprocess.run(hooks_cmd, capture_output=True, text=True)
    hooks_body, _, hooks_status = (hooks_result.stdout or "").rpartition("\n")
    hooks_body, hooks_status = hooks_body.strip(), hooks_status.strip()
    if hooks_status == "200":
        print("OpenClaw hooks test: PASS (HTTP 200)")
    else:
        hints = {
            "404": " — run section 3.6 to configure webhooks",
            "401": " — token mismatch; re-run section 3.6 to apply the current AGENT_HOOKS_TOKEN",
        }
        hint = hints.get(hooks_status, "")
        print(f"OpenClaw hooks test: FAIL (HTTP {hooks_status or 'unknown'}){hint}")
elif AGENT_RUNTIME == "hermes":
    print("OpenClaw hooks test skipped for Hermes runtime.")
else:
    print("OpenClaw hooks test skipped: AGENT_HOOKS_ENABLED is false.")

if gateway_container:
    def _sandbox_exec(sandbox_cmd):
        return [
            "openshell", "sandbox", "exec", "-n", NEMOCLAW_SANDBOX_NAME, "--",
            "sh", "-lc", sandbox_cmd,
        ]

    def _in_sandbox_show(label, sandbox_cmd):
        full = _sandbox_exec(sandbox_cmd)
        print(f"\n=== {label} ===")
        print("$", shlex.join(full))
        r = subprocess.run(full, capture_output=True, text=True, stdin=subprocess.DEVNULL)
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
        return r

    if AGENT_RUNTIME == "hermes":
        _in_sandbox_show("Hermes top-level docs", "ls -1 /sandbox/*.md 2>/dev/null || true")
        _in_sandbox_show(
            "VSS Orchestrator MCP policy path",
            "curl -s -o /dev/null --max-time 5 http://host.openshell.internal:9988/ && echo host alias reachable || true",
        )
    else:
        # _in_sandbox_show("openclaw plugins list", "openclaw plugins list")
        _in_sandbox_show("openclaw plugins doctor", "openclaw plugins doctor")

        skills_proc = subprocess.run(
            _sandbox_exec("openclaw skills list --json"),
            capture_output=True, text=True, stdin=subprocess.DEVNULL,
        )
        workspace_dir = None
        print("\n=== openclaw skills (non-bundled) ===")
        try:
            payload = json.loads(skills_proc.stdout)
            skills = payload["skills"] if isinstance(payload, dict) else payload
            if isinstance(payload, dict):
                workspace_dir = payload.get("workspaceDir")
            non_bundled = [s for s in skills if not s.get("bundled", False)]
            if non_bundled:
                for s in non_bundled:
                    print(f"- {s['name']}")
            else:
                print("No non-bundled OpenClaw skills found.")
        except (json.JSONDecodeError, TypeError) as exc:
            print(f"Could not parse skills list JSON: {exc}")
            if skills_proc.stderr:
                print(skills_proc.stderr)

        print(f"\n=== workspace .md files in {workspace_dir or '<unknown>'} ===")
        if workspace_dir:
            ls_cmd = f"ls -1 {shlex.quote(workspace_dir)} 2>/dev/null"
            ls_proc = subprocess.run(_sandbox_exec(ls_cmd), capture_output=True, text=True, stdin=subprocess.DEVNULL)
            present = {line.strip() for line in ls_proc.stdout.splitlines() if line.strip().endswith(".md")}
            for name in expected_workspace_md:
                mark = "OK  " if name in present else "MISS"
                print(f"  {mark}  {name}")
            extra = present - set(expected_workspace_md)
            if extra:
                print(f"  (also present: {sorted(extra)})")
        else:
            print("  workspaceDir not found in skills JSON payload; check skipped.")
else:
    print("Could not determine the OpenShell gateway container; runtime-specific workspace checks were skipped.")
